# Prompt Engineering for Math Problems - Kaggle Competition

## Introduction
This notebook is designed for the Kaggle competition on prompt engineering for solving TeX-formatted math tasks from 11th-grade exams. We will use the Groq API with the Llama-3.3-70B-Versatile model to generate answers. The focus is on crafting an effective prompt in English to achieve the best possible score.

Key steps:
- Load the test data with translations.
- Define a strong prompt template that encourages step-by-step reasoning.
- Use the Groq API to query the model for each problem.
- Parse the responses to extract numerical answers.
- Generate the submission.csv file.

Note: You need a Groq API key. Set it as an environment variable or input it when prompted.

In [7]:
# Cell 1
!pip install -q groq pandas tqdm

# Import libraries
import os
import pandas as pd
from groq import Groq
import re

In [8]:
# Load the test data with translations
# Assuming files are in /kaggle/input/ directory (standard for Kaggle notebooks)
test_df = pd.read_csv('/kaggle/input/prompt-engineering-math/test_with_translation.csv')

# Display the first few rows to verify
test_df.head()

,problem_id,problem_text,translation
0,11919,"Найдите значение выражения $4,8\cdot 2,5$.",Find the value of the expression $4.8\cdot 2.5$.
1,8513,Система навигации самолёта информирует пассажи...,The airplane's navigation system informs the p...
2,7887,Объём прямоугольного параллелепипеда вычисляет...,The volume of a rectangular parallelepiped is ...
3,5272,Найдите корень уравнения: $\left(\dfrac{1}{8} ...,Find the root of the equation: $\left(\dfrac{1...
4,8295,В школе есть двухместные туристические палатки...,"In the school, there are two-person tents. Wha..."


In [9]:
# Set up Groq client
# For Kaggle, use secrets: from kaggle_secrets import UserSecretsClient; user_secrets = UserSecretsClient(); api_key = user_secrets.get_secret("GROQ_API_KEY")
# Alternatively, set os.environ['GROQ_API_KEY'] = 'your_key_here' manually

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ['GROQ_API_KEY'] = "" # Fill this yourself

client = Groq()

In [10]:
# Define the system prompt (consistent across queries)
SYSTEM_PROMPT = """
You are a highly skilled math expert specializing in 11th-grade level problems, including algebra, geometry, calculus, and probability. Your goal is to solve the given math problem accurately.

Instructions:
- Read the problem carefully. It may include TeX formatting (e.g., \frac, \sqrt), but interpret it as mathematical expressions.
- Think step by step: Break down the problem, show your reasoning, and calculate precisely.
- Ensure the final answer is a single number: either an integer (e.g., 4231, -12) or a finite decimal (e.g., 0.75). Do not use commas, fractions, or extra text.
- Output the final answer in this exact format: \boxed{answer}
- Do not include any other text after the boxed answer.
"""

# Function to build user prompt for each problem
def build_user_prompt(translation):
    return f"""
Solve the following math problem step by step. The problem is provided in English translation, but it originates from TeX-formatted Russian text—treat any math symbols accordingly.

Problem: {translation}

Reason step by step, then provide the final numerical answer in \boxed{{}}.
"""

In [13]:
# Function to query the model and extract answer
def get_answer(translation):
    user_prompt = build_user_prompt(translation)
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.0,  # Low temperature for deterministic math solving
        max_tokens=1024,
        top_p=1.0
    )
    
    output = response.choices[0].message.content.strip()
    
    # Extract the boxed answer using regex
    match = re.search(r'\\boxed\{([-\d.]+)\}', output)
    if match:
        return match.group(1)
    else:
        # Fallback: Try to find the last number in the output
        numbers = re.findall(r'[-\d.]+', output)
        return numbers[-1] if numbers else "0"  # Default to 0 if parsing fails

# Generate answers for all test rows
answers = []
import time
for idx, row in test_df.iterrows():
    translation = row['translation']
    answer = get_answer(translation)
    answers.append(answer)
    print(f"Processed ID {row['problem_id']}: Answer = {answer}")
    time.sleep(1)
# Add answers to dataframe
test_df['answer'] = answers

Processed ID 11919: Answer = 12
Processed ID 8513: Answer = 11285
Processed ID 7887: Answer = 4
Processed ID 5272: Answer = 6
Processed ID 8295: Answer = 13
Processed ID 3219: Answer = 15
Processed ID 7235: Answer = 55
Processed ID 3688: Answer = 21
Processed ID 6116: Answer = .
Processed ID 4720: Answer = 3
Processed ID 12122: Answer = 3.
Processed ID 4311: Answer = 4
Processed ID 8283: Answer = 7
Processed ID 8347: Answer = 74
Processed ID 4170: Answer = 220
Processed ID 4: Answer = 21
Processed ID 8307: Answer = 10
Processed ID 7108: Answer = 544
Processed ID 7775: Answer = 800
Processed ID 7680: Answer = 216
Processed ID 8780: Answer = 1680
Processed ID 8707: Answer = 0.0225
Processed ID 7863: Answer = 0.8
Processed ID 12036: Answer = .
Processed ID 4569: Answer = 3
Processed ID 8134: Answer = 21
Processed ID 160: Answer = 9450
Processed ID 2869: Answer = 370.8
Processed ID 3309: Answer = 10
Processed ID 6941: Answer = 0.462
Processed ID 4880: Answer = 6
Processed ID 5126: Answer =

In [15]:
# Create submission dataframe
submission_df = test_df[['problem_id', 'answer']]

# Save to CSV
submission_df.to_csv('submission.csv', index=False)

# Display the first few rows
submission_df.head()

,problem_id,answer
0,11919,12
1,8513,11285
2,7887,4
3,5272,6
4,8295,13
